# Processamento de dados com scikit-learn

Este notebook apresenta ferramentas de preparação de dados que aparecem antes do treinamento de modelos. Ele segue a mesma separação explícita de treino e teste usada no módulo 001, sem `Pipeline` nesta etapa. A referência conceitual principal é o capítulo 2 de *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (Aurélien Géron, 2ª ed.).

## Regra central: evitar data leakage

Um imputer aprende mediana; um scaler aprende mínimo/máximo; um encoder aprende categorias. Esses valores devem ser aprendidos somente em `X_train`. Em seguida, aplique o mesmo objeto em `X_test` com `transform()`. O teste não participa de `fit()`.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, MaxAbsScaler, Normalizer,
    OneHotEncoder, OrdinalEncoder, PolynomialFeatures, FunctionTransformer, KBinsDiscretizer
)
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression

RANDOM_STATE = 42
pd.set_option('display.float_format', lambda value: f'{value:.3f}')

## 1. Dataset e `train_test_split`

Usamos `load_diabetes`, dataset incluído no scikit-learn. `test_size=0.2` reserva 20% para teste; `random_state=42` torna a divisão reproduzível. Em classificação, inclua `stratify=y` para preservar a proporção de classes. Para séries temporais, não embaralhe os dados.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data.copy()
y = diabetes.target.copy()

print('Observações:', X.shape[0], '| Features:', X.shape[1])
display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print('Treino:', X_train.shape, '| Teste:', X_test.shape)

## 2. `SimpleImputer`: valores ausentes

`SimpleImputer` preenche ausências com uma estatística aprendida no treino. Para features numéricas, `strategy='median'` é uma escolha frequente quando há assimetria ou outliers. Outras opções: `'mean'`, `'most_frequent'` e `'constant'`.

In [ ]:
# Criamos NaN apenas para demonstrar a ferramenta.
X_train_missing = X_train.copy()
X_test_missing = X_test.copy()
X_train_missing.loc[X_train_missing.index[:12], 'bmi'] = np.nan
X_test_missing.loc[X_test_missing.index[:3], 'bmi'] = np.nan

print('Ausências criadas no treino:', X_train_missing['bmi'].isna().sum())
print('Ausências criadas no teste:', X_test_missing['bmi'].isna().sum())

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train_missing), columns=X.columns, index=X_train.index
)

print('Mediana aprendida no treino para bmi:', imputer.statistics_[X.columns.get_loc('bmi')])

In [ ]:
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_missing), columns=X.columns, index=X_test.index
)

display(X_train_imputed[['bmi']].head())

### Variações: `KNNImputer` e `MissingIndicator`

`KNNImputer` usa observações vizinhas para estimar a ausência: `n_neighbors=5` é o padrão. Ele exige escala comparável e pode ser caro. `MissingIndicator` cria colunas booleanas que preservam a informação de que o valor estava ausente.

In [ ]:
indicator = MissingIndicator(features='missing-only')
missing_train = indicator.fit_transform(X_train_missing)
missing_test = indicator.transform(X_test_missing)

print('Features com indicador de ausência:', X.columns[indicator.features_].tolist())
print('Formato dos indicadores:', missing_train.shape, missing_test.shape)

# Para testar outra abordagem: KNNImputer(n_neighbors=5).fit_transform(X_train_missing)

## 3. `MinMaxScaler`: escala entre limites

`MinMaxScaler` transforma cada coluna para um intervalo, por padrão `[0, 1]`. É a escolha de escala usada no módulo 001 e deve ser aplicada antes de modelos sensíveis a distância ou regularização. O parâmetro `feature_range` pode ser, por exemplo, `(-1, 1)` ou `(0, 100)`. Outliers definem os extremos e podem comprimir os demais valores.

In [ ]:
minmax = MinMaxScaler()
X_train_minmax = pd.DataFrame(
    minmax.fit_transform(X_train_imputed), columns=X.columns, index=X_train.index
)

display(X_train_minmax.describe().loc[['min', 'max']])

In [ ]:
X_test_minmax = pd.DataFrame(
    minmax.transform(X_test_imputed), columns=X.columns, index=X_test.index
)

### Outras variações de escala

- `StandardScaler`: centraliza em média 0 e desvio padrão 1; útil para algoritmos que se beneficiam de features centradas.
- `MaxAbsScaler`: divide pelo maior valor absoluto e preserva zeros; adequado para matrizes esparsas.
- `Normalizer`: normaliza cada linha, não cada coluna; útil ao comparar vetores, como documentos.

Árvores e Random Forest normalmente não precisam de scaler, pois aprendem limites de divisão.

In [ ]:
# Exemplos de instanciação; use um por vez conforme o problema.
standard = StandardScaler()
max_abs = MaxAbsScaler()
normalizer = Normalizer(norm='l2')

print('MinMaxScaler, StandardScaler, MaxAbsScaler e Normalizer foram instanciados.')

## 4. Encoders para features categóricas

O dataset Diabetes é numérico; portanto, a coluna `faixa_bmi` abaixo é criada somente para demonstrar categorias. `OneHotEncoder` é apropriado para categorias nominais. `OrdinalEncoder` só deve ser usado quando existe ordem real entre as categorias.

In [ ]:
categorias_train = pd.DataFrame({
    'faixa_bmi': pd.cut(X_train['bmi'], bins=3, labels=['baixa', 'media', 'alta']).astype(str),
    'faixa_bp_ordenada': pd.cut(X_train['bp'], bins=3, labels=['baixa', 'media', 'alta']).astype(str),
}, index=X_train.index)

display(categorias_train.head())

In [ ]:
categorias_test = pd.DataFrame({
    'faixa_bmi': pd.cut(X_test['bmi'], bins=3, labels=['baixa', 'media', 'alta']).astype(str),
    'faixa_bp_ordenada': pd.cut(X_test['bp'], bins=3, labels=['baixa', 'media', 'alta']).astype(str),
}, index=X_test.index)

### `OneHotEncoder`

Cria uma coluna para cada categoria. `handle_unknown='ignore'` evita erro se uma categoria inédita aparecer no teste. `sparse_output=False` retorna um array denso apenas para facilitar a visualização; para muitas categorias, prefira a matriz esparsa padrão.

In [ ]:
one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
bmi_train_onehot = one_hot.fit_transform(categorias_train[['faixa_bmi']])

onehot_columns = one_hot.get_feature_names_out(['faixa_bmi'])
display(pd.DataFrame(bmi_train_onehot, columns=onehot_columns, index=X_train.index).head())

In [ ]:
bmi_test_onehot = one_hot.transform(categorias_test[['faixa_bmi']])

### `OrdinalEncoder`

Converta para números somente quando a ordem fizer sentido. Aqui, `baixa < media < alta` é uma ordem definida para a demonstração. Para uma categoria nominal como cidade, usar números 0, 1 e 2 criaria uma distância artificial; prefira One-Hot.

In [ ]:
ordinal = OrdinalEncoder(categories=[['baixa', 'media', 'alta']])
bp_train_ordinal = ordinal.fit_transform(categorias_train[['faixa_bp_ordenada']])

display(pd.DataFrame(bp_train_ordinal, columns=['faixa_bp_ordenada_codigo'], index=X_train.index).head())

In [ ]:
bp_test_ordinal = ordinal.transform(categorias_test[['faixa_bp_ordenada']])

## 5. Criação e seleção de features

`PolynomialFeatures` cria potências e interações para que um modelo linear represente curvas. `FunctionTransformer` aplica uma função conhecida, como `np.log1p`. `KBinsDiscretizer` cria faixas. Essas transformações aumentam ou alteram a representação dos dados; aplique-as apenas quando houver justificativa estatística ou de domínio.

In [ ]:
# Exemplos em duas features para evitar explosão de colunas.
duas_features_train = X_train_imputed[['bmi', 'bp']]
duas_features_test = X_test_imputed[['bmi', 'bp']]

poly = PolynomialFeatures(degree=2, include_bias=False)
poly_train = poly.fit_transform(duas_features_train)
poly_test = poly.transform(duas_features_test)
print('Features polinomiais:', poly.get_feature_names_out(['bmi', 'bp']).tolist())

In [ ]:
log_transform = FunctionTransformer(np.log1p, validate=False)
# Use log1p apenas em valores compatíveis com o domínio; não aplicamos ao dataset inteiro aqui.

bins = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
bmi_faixas_train = bins.fit_transform(duas_features_train[['bmi']])
bmi_faixas_test = bins.transform(duas_features_test[['bmi']])
print('Exemplo de faixas de bmi:', np.unique(bmi_faixas_train))

### Seleção de features

`VarianceThreshold` remove colunas constantes ou quase constantes sem olhar para `y`. `SelectKBest` escolhe features com base em sua associação univariada com o target. Como usa `y`, deve ser ajustado apenas no treino e repetido dentro da validação quando for feita seleção formal de modelos.

In [ ]:
variance = VarianceThreshold(threshold=0.0)
X_train_variance = variance.fit_transform(X_train_imputed)
X_test_variance = variance.transform(X_test_imputed)

print('Formato após remover features constantes:', X_train_variance.shape)

In [ ]:
selector = SelectKBest(score_func=f_regression, k=5)
X_train_selected = selector.fit_transform(X_train_imputed, y_train)
X_test_selected = selector.transform(X_test_imputed)

selected_features = X.columns[selector.get_support()].tolist()
print('Features selecionadas:', selected_features)
print('Formato após seleção:', X_train_selected.shape, X_test_selected.shape)

## Checklist antes de treinar

1. A divisão treino/teste respeita o tipo de dado (aleatório, temporal, grupo)?
2. Imputer, encoder, scaler e seletor foram ajustados somente no treino?
3. O encoder representa corretamente a existência — ou inexistência — de ordem entre categorias?
4. O scaler é necessário para o modelo escolhido?
5. Novas features e remoção de ausências são justificadas e reproduzíveis?